In [14]:
# Step 1: Install necessary libraries
#%pip install transformers datasets torch scikit-learn

In [15]:
# Step 2: Import and Load Model & Tokenizer
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
import torch

model_name = "distilbert-base-uncased"  # Good starting point for text classification
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)  # Binary classification
tokenizer = AutoTokenizer.from_pretrained(model_name)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


I'm using my own bank statement data. It is from July 16th, 2024 to January 7th, 2026.

In [16]:
#%pip install pandas numpy scikit-learn

In [17]:
import pandas as pd
import numpy as np

In [18]:

# Step 3: Load and Preprocess Your Dataset
# Assuming your data is in a CSV file
dataset = pd.read_csv('export_20260107.csv')

dataset

,Date,Description,Comments,Check Number,Amount,Balance
0,01/05/2026,ZEL ZELLE TO LIU BAOZHU,NaN,NaN,"-$1,750.00","$12,346.69"
1,01/02/2026,ECK VENMO 3264681992 BR PAYMENT,NaN,NaN,-$13.40,"$14,096.69"
2,12/31/2025,INT INTEREST CREDIT,NaN,NaN,$1.10,"$14,110.09"
3,12/31/2025,ECK VENMO 3264681992 BR PAYMENT,NaN,NaN,-$280.00,"$14,108.99"
4,12/24/2025,DIR FEDERAL NATIONAL 9111111101 BR PAYROLL,NaN,NaN,"$3,393.80","$14,388.99"
...,...,...,...,...,...,...
349,07/31/2024,INT INTEREST CREDIT,NaN,NaN,$0.83,"$10,766.46"
350,07/30/2024,ECK CITIZE CK WEBXFR 3770527921 BR ETRANSFER,NaN,NaN,-$20.00,"$10,765.63"
351,07/24/2024,DIR FANNIE MAE 9111111101 BR DIRECT DEP,NaN,NaN,"$1,332.94","$10,785.63"
352,07/22/2024,ECK Subscription 9000142694 BR Acorns,NaN,NaN,-$1.00,"$9,452.69"


In [19]:
labeled_dataset = dataset.copy()

labeled_dataset['rent'] = np.where(labeled_dataset['Description'].str.contains('LIU BAOZHU', case=False), 1, 0)

labeled_dataset.value_counts('rent')

rent
0    336
1     18
Name: count, dtype: int64

In [20]:
labeled_dataset

,Date,Description,Comments,Check Number,Amount,Balance,rent
0,01/05/2026,ZEL ZELLE TO LIU BAOZHU,NaN,NaN,"-$1,750.00","$12,346.69",1
1,01/02/2026,ECK VENMO 3264681992 BR PAYMENT,NaN,NaN,-$13.40,"$14,096.69",0
2,12/31/2025,INT INTEREST CREDIT,NaN,NaN,$1.10,"$14,110.09",0
3,12/31/2025,ECK VENMO 3264681992 BR PAYMENT,NaN,NaN,-$280.00,"$14,108.99",0
4,12/24/2025,DIR FEDERAL NATIONAL 9111111101 BR PAYROLL,NaN,NaN,"$3,393.80","$14,388.99",0
...,...,...,...,...,...,...,...
349,07/31/2024,INT INTEREST CREDIT,NaN,NaN,$0.83,"$10,766.46",0
350,07/30/2024,ECK CITIZE CK WEBXFR 3770527921 BR ETRANSFER,NaN,NaN,-$20.00,"$10,765.63",0
351,07/24/2024,DIR FANNIE MAE 9111111101 BR DIRECT DEP,NaN,NaN,"$1,332.94","$10,785.63",0
352,07/22/2024,ECK Subscription 9000142694 BR Acorns,NaN,NaN,-$1.00,"$9,452.69",0


In [21]:
from sklearn.model_selection import train_test_split
train_data, val_data = train_test_split(labeled_dataset, test_size=0.2, random_state=42, stratify=labeled_dataset['rent'])

In [23]:
labeled_dataset.to_csv('frank_labeled_data.csv', index=False)

train_data.to_csv('frank_train_data.csv', index=False)

val_data.to_csv('frank_val_data.csv', index=False)

In [10]:

# Step 3: Load and Preprocess Your Dataset
# Assuming your data is in a CSV file
dataset = load_dataset('csv', data_files={'train': 'frank_train_data.csv', 'validation': 'frank_val_data.csv'})

def preprocess_function(examples):
    # Tokenize the transaction descriptions. Use truncation and padding for uniform length.
    return tokenizer(examples['Description'], truncation=True, padding='max_length', max_length=128)

# Apply the tokenization to the entire dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Step 4: Format for PyTorch
# Rename the 'label' column to 'labels' (expected by the Trainer)
tokenized_dataset = tokenized_dataset.rename_column('rent', 'labels')
# Set the format to return PyTorch tensors
tokenized_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/283 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

In [11]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Function to compute accuracy and F1 score during evaluation."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')  # Use 'macro' for multi-class
    return {'accuracy': acc, 'f1': f1}

# Step 5: Define Training Arguments
training_args = TrainingArguments(
    output_dir='./rent_classifier_results',  # Where to save the model
    eval_strategy='epoch',             # Evaluate at the end of each epoch
    save_strategy='epoch',                   # Save a checkpoint each epoch
    learning_rate=2e-5,                      # Standard learning rate for fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,                      # Number of training epochs
    weight_decay=0.01,
    load_best_model_at_end=True,             # Load the best model at the end
    metric_for_best_model='f1',              # Use F1 score to choose the best model
    push_to_hub=False,                       # Set to True if you want to upload to Hugging Face Hub
)

# Step 6: Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Step 7: Train the Model
trainer.train()

/var/folders/8x/6jjxqd7n5gj_5yxg2wdp9gkw0000gn/T/ipykernel_7306/226895045.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.132332,0.943662,0.000000
2,No log,0.042620,1.000000,1.000000
3,No log,0.030125,1.000000,1.000000


TrainOutput(global_step=54, training_loss=0.16001950369940865, metrics={'train_runtime': 52.2534, 'train_samples_per_second': 16.248, 'train_steps_per_second': 1.033, 'total_flos': 28116205364736.0, 'train_loss': 0.16001950369940865, 'epoch': 3.0})

In [13]:
# Evaluate the model on the validation set
eval_results = trainer.evaluate()

# Print the evaluation results
print(eval_results)

{'eval_loss': 0.04262031987309456, 'eval_accuracy': 1.0, 'eval_f1': 1.0, 'eval_runtime': 1.3421, 'eval_samples_per_second': 52.902, 'eval_steps_per_second': 3.725, 'epoch': 3.0}
